# 0. 저장 경로 지정

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

dataset_path = '/content/drive/MyDrive/Colab Notebooks/26_Deeplearning/딥러닝 팀플/데이터/dataset'

if not os.path.exists(dataset_path):
    os.makedirs(dataset_path)
    print(f"'{dataset_path}' 경로를 생성했습니다.")
else:
    print(f"'{dataset_path}' 경로는 이미 존재합니다.")

%cd $dataset_path

'/content/drive/MyDrive/Colab Notebooks/26_Deeplearning/딥러닝 팀플/데이터/dataset' 경로를 생성했습니다.
/content/drive/MyDrive/Colab Notebooks/26_Deeplearning/딥러닝 팀플/데이터/dataset


# 1. AIHUB 데이터 저장 (실패)
코랩 서버가 국외에 있어서 실패. 노트북 용량이 부족해서, 집컴에서 로컬로 다운로드 받고 업로드 할게요

In [3]:
!curl -o "aihubshell" https://api.aihub.or.kr/api/aihubshell.do
!chmod +x aihubshell

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  7824  100  7824    0     0   6693      0  0:00:01  0:00:01 --:--:--  6698


In [4]:
# aihubshell 실행 파일 경로 정의
shell_bin = './aihubshell'

In [5]:
#api 키 불러오기
from google.colab import userdata
AIHUB_API_KEY = userdata.get('aihub_api_key')

SecretNotFoundError: Secret aihub_api_key does not exist.

In [6]:
# 다운로드 대상 설정 (카테고리명: [파일키 리스트])
# 예시: 폭행 데이터 키 범위를 리스트로 생성
assault_indoor = [str(i) for i in range(49825, 49835)]
assault_outdoor = [str(i) for i in range(49841, 49851)]
burglary_indoor = [str(i) for i in range(49765 , 49768 )]
burglary_outdoor = [str(i) for i in range(49772 , 49781 )]
vandalism_indoor = [str(49782)]
vandalism_outdoor = [str(i) for i in range(49784 , 49791 )]
swoon_indoor = [str(i) for i in range(49792 , 49798 )]
swoon_outdoor = [str(i) for i in range(49803 , 49702 )]


file_map = {
    "01.폭행/실내_원본": assault_indoor,
    "01.폭행/실외": assault_outdoor,
    "03.절도/실내_원본": burglary_indoor,
    "03.절도/실외": burglary_outdoor,
    "04.기물파손/실내_원본": vandalism_indoor,
    "04.기물파손/실외": vandalism_outdoor,
    "05.실신/실내_원본": swoon_indoor,
    "05.실신/실외": swoon_outdoor
}

dataset_key = "171"

for folder_name, keys in file_map.items():
    if not keys or "추가확인" in keys[0]:
        continue

    print(f"\n>>> {folder_name} 다운로드 시작 (파일 수: {len(keys)}개)")

    # 콤마로 구분된 키 문자열 생성
    key_str = ",".join(keys)

    # 다운로드 실행 (해당 폴더 내부에서 실행)
    !{shell_bin} -mode d -datasetkey {dataset_key} -filekey {key_str} -aihubapikey '{AIHUB_API_KEY}'

print("\n✨ 지정된 모든 데이터 다운로드 완료!")


>>> 01.폭행/실내_원본 다운로드 시작 (파일 수: 10개)
/bin/bash: line 1: {shell_bin}: command not found

>>> 01.폭행/실외 다운로드 시작 (파일 수: 10개)
/bin/bash: line 1: {shell_bin}: command not found

>>> 03.절도/실내_원본 다운로드 시작 (파일 수: 3개)
/bin/bash: line 1: {shell_bin}: command not found

>>> 03.절도/실외 다운로드 시작 (파일 수: 9개)
/bin/bash: line 1: {shell_bin}: command not found

>>> 04.기물파손/실내_원본 다운로드 시작 (파일 수: 1개)
/bin/bash: line 1: {shell_bin}: command not found

>>> 04.기물파손/실외 다운로드 시작 (파일 수: 7개)
/bin/bash: line 1: {shell_bin}: command not found

>>> 05.실신/실내_원본 다운로드 시작 (파일 수: 6개)
/bin/bash: line 1: {shell_bin}: command not found

✨ 지정된 모든 데이터 다운로드 완료!


# 2. Kaggle 데이터 저장 (완료)

In [7]:
import sys
!{sys.executable} -m pip install kagglehub

In [8]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("odins0n/ucf-crime-dataset")

print("Path to dataset files:", path)

100%|██████████| 11.0G/11.0G [02:02<00:00, 96.6MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/odins0n/ucf-crime-dataset/versions/1


In [ ]:
import shutil
import os

# Source path where kagglehub downloaded the data
source_path = path

# Destination path in Google Drive
destination_path = dataset_path # This variable was defined earlier

# Check if the source directory exists
if os.path.exists(source_path):
    # Get all items (files and folders) in the source directory
    items = os.listdir(source_path)

    # Iterate over each item and copy it to the destination
    for item in items:
        s = os.path.join(source_path, item)
        d = os.path.join(destination_path, item)
        if os.path.isdir(s):
            shutil.copytree(s, d, dirs_exist_ok=True) # Use dirs_exist_ok for folders
        else:
            shutil.copy2(s, d) # Use copy2 to preserve metadata
    print(f"Successfully copied all files from '{source_path}' to '{destination_path}'")
else:
    print(f"Source path '{source_path}' does not exist.")